# Fruit Vision – אימון v3 (Transfer Learning)

מה שונה מהגרסה הקודמת ולמה:

| שינוי | הסיבה |
|---|---|
| **MobileNetV2 מאומן מראש** במקום CNN קטן מאפס | המודל כבר מכיר רקעים, תאורה ומרקמים אמיתיים מ-ImageNet |
| **נתוני `Fruit_v3`** (תמונות מעבדה + תמונות אמיתיות) | הגיוון הוא מה שהיה חסר, לא הכמות |
| **החלפת רקע לבן ברקעים אקראיים** בזמן האימון | המודל למד "רקע בז' = בננה" – כך הוא מפסיק להסתמך על הרקע |
| **ולידציה לפי קבוצות** (זן / תמונת מקור) | ב-v2 תמונות כמעט זהות היו גם באימון וגם בוולידציה, ולכן 100% מטעה |
| **משקל x3 לתמונות אמיתיות** + משקל למחלקות | יש הרבה פחות תמונות אמיתיות מתמונות מעבדה |
| **Label Smoothing + Dropout + Early Stopping** | פחות ביטחון מופרז בתשובות שגויות, פחות התאמת יתר |

**לפני הרצה:** `Runtime ← Change runtime type ← GPU`, והעלה את `Fruit_v3.zip` ל-Drive (הנתיב בתא 2).

In [ ]:
import os, sys, csv, json, random, shutil, zipfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SMOKE = False            # True = ריצה קצרה לבדיקה בלבד (לא להשתמש לאימון אמיתי)
SEED, IMG, BATCH = 42, 224, 32
CLASS_NAMES = ['Apple', 'Avocado', 'Banana', 'Eggplant', 'Tomato']   # סדר אלפביתי = סדר class_names.json הקיים
REAL_WEIGHT = 3.0        # משקל לתמונות אמיתיות (טלפון / אינטרנט) לעומת תמונות מעבדה
IN_COLAB = 'google.colab' in sys.modules

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# --- טעינת הנתונים. מוחקים חילוץ קודם כדי לא לאמן בטעות על נתונים ישנים (זה מה שקרה בפעם הקודמת) ---
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/מכללת איקום/Fruit_v3.zip'    # <-- תקן אם שמרת במקום אחר
    shutil.rmtree('/content/dataset', ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content/dataset')
    DATA_DIR = '/content/dataset/Fruit_v3'
else:
    DATA_DIR = r'C:\Users\Matan\Downloads\Fruit_v3'

rows = []
with open(os.path.join(DATA_DIR, 'manifest.csv'), encoding='utf-8') as f:
    for r in csv.DictReader(f):
        r['path'] = os.path.join(DATA_DIR, r['split'], r['class'], r['file'])
        r['is_lab'] = (r['origin'] == 'fruits360')      # תמונות מעבדה על רקע לבן
        rows.append(r)

missing = [r['path'] for r in rows if not os.path.exists(r['path'])]
print(len(rows), 'files in manifest | missing:', len(missing))
assert not missing, missing[:3]
assert sorted({r['class'] for r in rows}) == CLASS_NAMES

In [ ]:
# --- חלוקה: אימון / ולידציה (לפי קבוצות!) / בדיקה ---
train_rows = [r for r in rows if r['split'] == 'train']
test_rows  = [r for r in rows if r['split'] == 'test']

# ולידציה: מוציאים קבוצות שלמות (זן או תמונת מקור), לא תמונות בודדות.
# תמונות רצופות של אותו פרי כמעט זהות, ואם יתפצלו בין אימון לוולידציה התוצאה תהיה מנופחת.
rng = random.Random(SEED)
val_sources = set()
for c in CLASS_NAMES:
    rc = [r for r in train_rows if r['class'] == c]
    n_lab_groups = len({r['source'] for r in rc if r['is_lab']})
    # למחלקה עם מעט זנים (חציל, בננה, אבוקדו) לא מוציאים זן שלם מהאימון – הוולידציה מתמונות אמיתיות
    pool = sorted({r['source'] for r in rc if not r['is_lab']}) if n_lab_groups < 8 else sorted({r['source'] for r in rc})
    for s in rng.sample(pool, max(1, round(0.15 * len(pool)))):
        val_sources.add((c, s))

val_rows = [r for r in train_rows if (r['class'], r['source']) in val_sources]
fit_rows = [r for r in train_rows if (r['class'], r['source']) not in val_sources]

if SMOKE:   # רק לבדיקה מהירה
    fit_rows = [r for c in CLASS_NAMES for r in [x for x in fit_rows if x['class'] == c][:30]]
    val_rows, test_rows = val_rows[:50], test_rows[:80]

def summary(name, rs):
    print(f'{name:10s} total {len(rs):5d} | real-world {sum(not r["is_lab"] for r in rs):4d} | ' +
          ' '.join(f'{c}:{sum(r["class"] == c for r in rs)}' for c in CLASS_NAMES))
summary('fit', fit_rows); summary('validation', val_rows); summary('test', test_rows)

# משקלים: מחלקות נדירות + תמונות אמיתיות
def raw_w(r): return 1.0 if r['is_lab'] else REAL_WEIGHT
tot = {c: sum(raw_w(r) for r in fit_rows if r['class'] == c) for c in CLASS_NAMES}
class_w = {c: sum(tot.values()) / (len(CLASS_NAMES) * tot[c]) for c in CLASS_NAMES}
def sample_weight(r): return class_w[r['class']] * raw_w(r)
print('class weights:', {c: round(v, 2) for c, v in class_w.items()})

In [ ]:
# --- צנרת נתונים: טעינה, החלפת רקע, אוגמנטציה ---
AUTOTUNE = tf.data.AUTOTUNE

def load(path, y, lab, w):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    x = tf.image.resize(x, (IMG, IMG))                      # float32 בטווח 0..255
    return x, tf.one_hot(y, len(CLASS_NAMES)), lab, w

def background_mask(x):
    # מחזיר 1 היכן שיש רקע לבן שמחובר לשולי התמונה (ולא לבן שבתוך הפרי, כמו ברק על עגבנייה)
    small = tf.image.resize(x, (56, 56), antialias=True)
    mx, mn = tf.reduce_max(small, -1, keepdims=True), tf.reduce_min(small, -1, keepdims=True)
    white = tf.cast((mn > 0.86) & ((mx - mn) < 0.14), tf.float32)
    border = tf.pad(tf.zeros_like(white[:, 1:-1, 1:-1, :]), [[0, 0], [1, 1], [1, 1], [0, 0]], constant_values=1.)
    seed = border * white
    for _ in range(40):                                     # "שפיכה" של הרקע מהשוליים פנימה
        seed = tf.nn.max_pool2d(seed, 3, 1, 'SAME') * white
    m = tf.image.resize(seed, (IMG, IMG), method='bilinear')
    # מסירים את ה"הילה" הלבנה סביב הפרי: פיקסלים כמעט-לבנים שצמודים לרקע נחשבים רקע
    mx2, mn2 = tf.reduce_max(x, -1, keepdims=True), tf.reduce_min(x, -1, keepdims=True)
    near_white = tf.cast((mn2 > 0.78) & ((mx2 - mn2) < 0.22), tf.float32)
    m = tf.maximum(m, tf.nn.max_pool2d(m, 5, 1, 'SAME') * near_white)
    return tf.nn.avg_pool2d(m, 3, 1, 'SAME')                # קצה רך

# רקעים "ריאליסטיים": גוני בז', אפור, עץ, שחור, לבן, ירוק כהה – בלי צבעי קשת
PALETTE = tf.constant([[.82, .78, .70], [.62, .62, .64], [.45, .32, .20], [.10, .10, .10],
                       [.95, .95, .95], [.30, .42, .25], [.70, .55, .40], [.35, .35, .38]])

def random_background(b):
    pick = tf.gather(PALETTE, tf.random.uniform([b], 0, tf.shape(PALETTE)[0], dtype=tf.int32))[:, None, None, :]
    pick += tf.random.uniform([b, 1, 1, 3], -0.05, 0.05)
    gray = tf.random.uniform([b, 1, 1, 1], 0.10, 0.95) + tf.random.uniform([b, 1, 1, 3], -0.06, 0.06)   # אפור עם גוון קל
    use_palette = tf.cast(tf.random.uniform([b, 1, 1, 1]) < 0.8, tf.float32)
    base = pick * use_palette + gray * (1 - use_palette)
    low = tf.image.resize(tf.random.normal([b, 6, 6, 1], 0, 0.10), (IMG, IMG), method='bicubic')   # כתמי בהירות רכים
    grain = tf.random.normal([b, IMG, IMG, 1], 0, 0.05)                                           # גרגירים כמו משטח
    return tf.clip_by_value(base + low + grain, 0, 1)

# אוגמנטציה גיאומטרית: האזורים החדשים מתמלאים בלבן, ואז הם נחשבים רקע ומוחלפים (אין עותקים משוקפים של הפרי)
geometric = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15, fill_mode='constant', fill_value=1.0),
    layers.RandomZoom((-0.3, 0.2), fill_mode='constant', fill_value=1.0),
    layers.RandomTranslation(0.1, 0.1, fill_mode='constant', fill_value=1.0),
])
photometric = keras.Sequential([
    layers.RandomBrightness(0.2, value_range=(0, 1)),
    layers.RandomContrast(0.2),
])

def train_transform(x, y, lab, w):
    x = geometric(x / 255.0, training=True)
    b = tf.shape(x)[0]
    m = background_mask(x)
    p = tf.where(lab, 0.85, 0.5)                            # תמונות אמיתיות: מוחלף בעיקר שוליים לבנים
    use = tf.cast(tf.random.uniform([b]) < p, tf.float32)[:, None, None, None]
    x = x * (1 - m * use) + random_background(b) * (m * use)
    x = photometric(x, training=True)
    return tf.clip_by_value(x, 0, 1), y, w

def make_ds(rs, training):
    ds = tf.data.Dataset.from_tensor_slices((
        tf.constant([r['path'] for r in rs]),
        tf.constant([CLASS_NAMES.index(r['class']) for r in rs]),
        tf.constant([r['is_lab'] for r in rs]),
        tf.constant([sample_weight(r) for r in rs], dtype=tf.float32)))
    if training:
        ds = ds.shuffle(len(rs), seed=SEED)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH)
    ds = ds.map(train_transform, num_parallel_calls=AUTOTUNE) if training else ds.map(lambda x, y, l, w: (x / 255.0, y))
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(fit_rows, True), make_ds(val_rows, False), make_ds(test_rows, False)

In [ ]:
# --- בדיקה ויזואלית: איך נראים נתוני האימון אחרי החלפת רקע ואוגמנטציה? ---
import matplotlib.pyplot as plt
xb, yb, _ = next(iter(train_ds))
plt.figure(figsize=(14, 7))
for i in range(min(16, len(xb))):
    plt.subplot(2, 8, i + 1); plt.imshow(xb[i].numpy()); plt.axis('off')
    plt.title(CLASS_NAMES[int(np.argmax(yb[i]))], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# --- המודל: MobileNetV2 מאומן מראש. הנרמול בתוך המודל, כך ששרת ה-API ממשיך לשלוח תמונה בטווח 0..1 ---
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = keras.Input((IMG, IMG, 3))                       # קלט: 0..1 (כמו שהשרת שולח)
x = layers.Rescaling(2.0, offset=-1.0)(inputs)            # ל-0..1 → -1..1 כפי ש-MobileNetV2 מצפה
x = base(x, training=False)                               # BatchNorm נשאר במצב הסקה גם בשלב ה-fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = keras.Model(inputs, outputs)

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=['accuracy'])

callbacks = [keras.callbacks.EarlyStopping('val_loss', patience=3, restore_best_weights=True),
             keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=1)]

# שלב 1: מאמנים רק את הראש החדש
compile_model(1e-3)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=1 if SMOKE else 10, callbacks=callbacks)

In [ ]:
# שלב 2: Fine-tuning – משחררים את השכבות העליונות של MobileNetV2 עם קצב למידה קטן מאוד
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
compile_model(1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=1 if SMOKE else 12, callbacks=callbacks)

In [ ]:
# --- הערכה על סט הבדיקה: בנפרד לתמונות מעבדה ולתמונות אמיתיות ---
probs = model.predict(test_ds, verbose=0)
y_true = np.array([CLASS_NAMES.index(r['class']) for r in test_rows])
pred, conf = probs.argmax(1), probs.max(1)
lab = np.array([r['is_lab'] for r in test_rows])

def report(mask, title):
    if mask.sum() == 0: return
    yt, yp, cf = y_true[mask], pred[mask], conf[mask]
    print(f'\n=== {title}: {mask.sum()} תמונות')
    print(f'דיוק: {(yt == yp).mean():.1%} | דיוק באפליקציה (מתחת ל-60% = unknown = שגיאה): {((yt == yp) & (cf >= .6)).mean():.1%}')
    print('לפי פרי:', '  '.join(f'{c} {((yp == i) & (yt == i)).sum()}/{(yt == i).sum()}' for i, c in enumerate(CLASS_NAMES) if (yt == i).sum()))
    wrong = yp != yt
    print(f'טעויות בביטחון >=90%: {(wrong & (cf >= .9)).sum()} מתוך {wrong.sum()} טעויות')

report(np.ones(len(test_rows), bool), 'כל סט הבדיקה')
report(~lab, 'תמונות אמיתיות (טלפון/אינטרנט)  <-- זה המספר החשוב')
report(lab, 'תמונות מעבדה (רקע לבן)')

cm = np.zeros((5, 5), int)
for t, p in zip(y_true, pred): cm[t, p] += 1
print('\nמטריצת בלבול (שורה = האמת, עמודה = הניבוי):')
print('          ' + ' '.join(f'{c[:7]:>8s}' for c in CLASS_NAMES))
for i, c in enumerate(CLASS_NAMES): print(f'{c:9s} ' + ' '.join(f'{v:8d}' for v in cm[i]))

In [ ]:
# --- (אופציונלי) בדיקה על התמונות שלך: תיקייה my_test ב-Drive עם תת-תיקיות Apple, Avocado, Banana, Eggplant, Tomato ---
MY_TEST_DIR = '/content/drive/MyDrive/my_test'
if os.path.isdir(MY_TEST_DIR):
    items = [(os.path.join(MY_TEST_DIR, c, f), c) for c in CLASS_NAMES if os.path.isdir(os.path.join(MY_TEST_DIR, c))
             for f in os.listdir(os.path.join(MY_TEST_DIR, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    ok = 0
    for path, c in items:
        img = tf.image.resize(tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False), (IMG, IMG)) / 255.0
        p = model.predict(img[None], verbose=0)[0]
        label = CLASS_NAMES[int(p.argmax())] if p.max() >= .6 else 'unknown'
        ok += label == c
        print(f'{os.path.basename(path):30s} אמת={c:9s} ניבוי={label:9s} ({p.max():.0%})', '' if label == c else '  <-- שגיאה')
    print(f'\nהצלחה על התמונות שלך: {ok}/{len(items)}')
else:
    print('אין תיקיית my_test – מדלגים.')

In [ ]:
# --- שמירה: קבצים חדשים בשמות חדשים, בלי לדרוס את הקודמים ---
OUT_DIR = '/content/drive/MyDrive/vision-project' if IN_COLAB else os.path.join(os.getcwd(), 'out_v3')
os.makedirs(OUT_DIR, exist_ok=True)
model.save(os.path.join(OUT_DIR, 'model_v3.keras'))
with open(os.path.join(OUT_DIR, 'class_names_v3.json'), 'w', encoding='utf-8') as f:
    json.dump(CLASS_NAMES, f)
print('נשמר ב:', OUT_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(OUT_DIR, 'model_v3.keras'))
    files.download(os.path.join(OUT_DIR, 'class_names_v3.json'))